# Kaggle Competition info:
## NVIDIA Nemotron Model Reasoning Challenge
#### Advance reasoning techniques using NVIDIA Nemotron open models on a novel benchmark

## Overview
Develop techniques that improve reasoning accuracy using NVIDIA Nemotron models.

Participants will experiment with prompting, data pipelines, and lightweight fine-tuning while evaluating their approaches on a new reasoning benchmark developed by NVIDIA Research.

**Link:** <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/overview>

## Install libary

In [1]:
install_DIR= "kaggle/input/datasets/johnsonhk88"
# Install deepeval package
!pip install deepeval  --no-index --no-deps --find-links=file:////{install_DIR}/deepeval-4-0-2/deepeval-package/deepeval-4.0.2-py3-none-any.whl
!pip install portalocker --no-index --no-deps --find-links=file:////{install_DIR}/deepeval-4-0-2/deepeval-package/portalocker-3.2.0-py3-none-any.whl
!pip install posthog --no-index --no-deps --find-links=file:////{install_DIR}/deepeval-4-0-2/deepeval-package/posthog-7.15.0-py3-none-any.whl
!pip install pyfiglet --no-index --no-deps --find-links=file:////{install_DIR}/deepeval-4-0-2/deepeval-package/pyfiglet-1.0.4-py3-none-any.whl

# Install Mlflow
!pip install mlflow --no-index --no-deps --find-links=file:////{install_DIR}/mlflow/mlflow-package/mlflow-3.12.0-py3-none-any.whl
!pip install mlflow_skinny --no-index --no-deps --find-links=file:////{install_DIR}/mlflow/mlflow-package/mlflow_skinny-3.12.0-py3-none-any.whl
!pip install mlflow_tracing --no-index --no-deps --find-links=file:////{install_DIR}/mlflow/mlflow-package/mlflow_tracing-3.12.0-py3-none-any.whl
!pip install databricks-sdk --no-index --no-deps --find-links=file:////{install_DIR}/mlflow/mlflow-package/databricks_sdk-0.110.0-py3-none-any.whl

# !pip install --no-build-isolation -q "mamba-ssm>=2.2" causal-conv1d
!pip install causal-conv1d  --no-index --no-deps --find-links=file:////kaggle/input/datasets/grooking/mamba-ssm/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl
!pip install mamba-ssm  --no-index --no-deps --find-links=file:////kaggle/input/datasets/grooking/mamba-ssm/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl

!pip install bitsandbytes --no-index --no-deps --find-links=file:////kaggle/input/datasets/johnsonhk88/bitsandbytes/bitsandbytes-package/bitsandbytes-0.48.0-py3-none-manylinux_2_24_x86_64.whl

Looking in links: file:////kaggle/input/datasets/johnsonhk88/deepeval-4-0-2/deepeval-package/deepeval-4.0.2-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/deepeval-4-0-2/deepeval-package/portalocker-3.2.0-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/deepeval-4-0-2/deepeval-package/posthog-7.15.0-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/deepeval-4-0-2/deepeval-package/pyfiglet-1.0.4-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/mlflow/mlflow-package/mlflow-3.12.0-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/mlflow/mlflow-package/mlflow_skinny-3.12.0-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/mlflow/mlflow-package/mlflow_tracing-3.12.0-py3-none-any.whl
Looking in links: file:////kaggle/input/datasets/johnsonhk88/mlflow/mlflow-package/databricks_sdk-0.110.0-py3-none-any.whl
Looking in links:

In [2]:
# !pip install posthog==7.15.0
# !pip install portalocker==3.2.0
# !pip install pyfiglet==1.0.4
# !pip install databricks-sdk==0.110.0

In [3]:
# !pip install -q triton
# !pip install --no-build-isolation -q "mamba-ssm>=2.2" causal-conv1d

In [4]:
import triton
import mamba_ssm
import bitsandbytes

print(triton.__version__)
print(mamba_ssm.__version__)
print(bitsandbytes.__version__)

3.6.0
2.3.1
0.48.0


In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
import deepeval
import mlflow

In [6]:
from deepeval.metrics import GEval                     # ← Only this is needed
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

In [7]:
# from trl import DPOTrainer, DPOConfig
# from peft import PeftModel, LoraConfig
# from datasets import load_dataset
# import torch

import os, glob, sys, subprocess, site, importlib.util, shutil, stat, types, re

import datasets
import kagglehub

import torch
# import mamba_ssm later

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

import pandas as pd
import random
import gc, time

import json, zipfile

In [8]:
import pandas as pd
import torch
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from typing import Optional, Tuple, Dict


import kagglehub
# import mamba_ssm
# from trl import GRPOConfig, GRPOTrainer

In [9]:
class CFG:
    trainFile = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
    testFile = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv"

    llmModel1 = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

In [10]:
mlflow_local= True # suitable for kaggle envirorment
OUTPUT_DIR = "./model-finetuned"
MLFLOW_EXPERIMENT = "Nemoton-Reasoning-Fine-Tuning"


if mlflow_local:
    mlflowUrl = "sqlite:///mlflow.db" # for local sqlite db (recommend)
    # mlflowUrl = "file:///mlruns" # for local file 

else:
    # localhost
    mlflowUrl = "http://localhost:5000"




def setup_mlflow_run(
    experiment_name: str = "wonderland_nemotron_reasoning",
    run_name_suffix: str = "",
    train: bool = True
):
    """
    Separate MLflow setup function.
    Sets tracking URI, experiment, starts a run, and returns the active run context.
    """
    # project setup
    mlflow.set_tracking_uri(mlflowUrl)
    mlflow.set_experiment(experiment_name)

    # === CRITICAL: End any previously active run to prevent conflict ===
    try:
        mlflow.end_run()          # safely ends any lingering run
    except:
        pass                      # no active run → ignore
    
    run_name = f"eval_{run_name_suffix}"
    run = mlflow.start_run(run_name=run_name)
    print(f"MLflow run started: {run_name} (ID: {run.info.run_id})")
    return run

    if train:
        run_name = f"train_{run_name_suffix}"
    else:
        run_name = f"eval_{run_name_suffix}"
    run = mlflow.start_run(run_name=run_name)
    print(f"MLflow run started: {run_name} (ID: {run.info.run_id})")
    return run
        
    


## Load Dataset

In [11]:
# ====================== DATASET LOADING ======================
def load_train_data() -> pd.DataFrame:
    """Load the full training dataset (train.csv)"""
    df = pd.read_csv(CFG.trainFile)
    print(f"✅ Loaded train dataset: {len(df):,} examples")
    return df


def load_val_data(val_size: int = 400) -> pd.DataFrame:
    """Load a fixed validation split from train.csv (same samples every run)"""
    train_df = load_train_data()
    val_df = train_df.sample(n=val_size, random_state=42).reset_index(drop=True)
    print(f"✅ Created fixed validation split: {len(val_df)} examples")
    return  train_df , val_df


def load_test_data() -> pd.DataFrame:
    """Load the official test dataset (test.csv) for final inference"""
    df = pd.read_csv(CFG.testFile)
    print(f"✅ Loaded test dataset: {len(df)} examples")
    return df

In [12]:
trainDF, valDF = load_val_data(1000)


✅ Loaded train dataset: 9,500 examples
✅ Created fixed validation split: 1000 examples


In [13]:
# trainDF
valDF

,id,prompt,answer
0,18c797f1,"In Alice's Wonderland, a secret bit manipulati...",11100111
1,b13d511a,"In Alice's Wonderland, a secret set of transfo...",\&[[
2,71cd0e14,"In Alice's Wonderland, numbers are secretly co...",XXV
3,452b0241,"In Alice's Wonderland, a secret unit conversio...",20.72
4,7a4063e6,"In Alice's Wonderland, a secret bit manipulati...",10010110
...,...,...,...
995,56b1e037,"In Alice's Wonderland, the gravitational const...",10.66
996,492fa295,"In Alice's Wonderland, secret encryption rules...",princess found the dark crystal
997,d5aa5564,"In Alice's Wonderland, a secret set of transfo...",45
998,10ff9431,"In Alice's Wonderland, a secret set of transfo...",0271


In [14]:
testDF =  load_test_data()
testDF

✅ Loaded test dataset: 3 examples


,id,prompt
0,00066667,"In Alice's Wonderland, a secret bit manipulati..."
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati..."
2,00189f6a,"In Alice's Wonderland, secret encryption rules..."


In [15]:
print(trainDF["prompt"][0])

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100


# Load Model

In [16]:
BASE_MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
BASE_MODEL_PATH

'/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'

In [18]:
def load_model(
    model_name: str,
    adapter_path: Optional[str] = None
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load base model + optional LoRA adapter."""
    print(f"Loading model: {model_name} {'+ adapter: ' + adapter_path if adapter_path else '(base)'}")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )

    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

## Evaluation function

In [32]:
def run_evaluation(
    model,
    tokenizer,
    val_size: int = 400,
    experiment_name: str = "wonderland_nemotron_reasoning",
    run_name_suffix: str = ""
) -> Dict:
    run = setup_mlflow_run(experiment_name, run_name_suffix)

    mlflow.log_param("val_size", val_size)
    mlflow.log_param("generation_method", "apply_chat_template + greedy_generate")

    test_cases = []
    exact_matches = []
    start_time = time.time()

    print(f"Running evaluation on {val_size} samples using direct generate...")

    for i, row in valDF.iterrows():
        prompt = row['prompt']
        gt = str(row['answer']).strip()

        # messages = [
        #     {"role": "system", "content": "You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles. Analyze the examples carefully and output ONLY the final answer."},
        #     {"role": "user", "content": prompt}
        # ]

        # # === ROBUST apply_chat_template for Nemotron-30B-A3B ===
        # tokenized = tokenizer.apply_chat_template(
        #     messages,
        #     tokenize=True,
        #     add_generation_prompt=True,
        #     return_tensors="pt"
        # )

        # # Force correct input_ids (this is the fix)
        # if isinstance(tokenized, dict):
        #     input_ids = tokenized["input_ids"]
        #     attention_mask = tokenized.get("attention_mask", None)
        # else:
        #     input_ids = tokenized
        #     attention_mask = None

        # # Ensure it's a tensor and on the right device
        # input_ids = input_ids.to(model.device)
        # if attention_mask is not None:
        #     attention_mask = attention_mask.to(model.device)

        # === Generate ===
        # outputs = model.generate(
        #     input_ids,
        #     attention_mask=attention_mask,
        #     max_new_tokens=256,
        #     do_sample=False,
        #     num_beams=1,
        #     eos_token_id=tokenizer.eos_token_id,
        #     pad_token_id=tokenizer.pad_token_id,
        # )

        full_prompt = f"""<|system|>You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles. Analyze the examples carefully and output ONLY the final answer.</s>
<|user|>{prompt}</s>
<|assistant|>"""

        # Tokenize as string → tensor
        tokenized = tokenizer(
            full_prompt,
            return_tensors="pt",
            add_special_tokens=True
        )

        input_ids = tokenized.input_ids.to(model.device)
        attention_mask = tokenized.attention_mask.to(model.device)

        # === Generate ===
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        
        # === CLEAN DECODING (new) ===
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        # Extract only assistant response
        if "<|assistant|>" in full_output:
            pred = full_output.split("<|assistant|>")[-1]
        else:
            pred = full_output
        
        # Remove common special tokens
        special_tokens = ["<|im_end|>", "<|endoftext|>", "</s>", "<|eot_id|>", "<|im_start|>"]
        for token in special_tokens:
            pred = pred.replace(token, "").strip()
        
        pred = pred.strip()   # final clean

        is_exact = 1.0 if pred.lower().strip() == gt.lower().strip() else 0.0
        exact_matches.append(is_exact)

        test_case = LLMTestCase(input=prompt, actual_output=pred, expected_output=gt)
        test_cases.append(test_case)
        print(f"Test: {i}")

    # Metrics
    avg_exact_match = sum(exact_matches) / len(exact_matches)
    mlflow.log_metric("exact_match", avg_exact_match)
    mlflow.log_metric("eval_time_sec", time.time() - start_time)

    # DeepEval
    correctness_metric = GEval(
        name="Answer Correctness",
        criteria="Determine whether the actual output exactly matches the expected output. Be strict.",
        evaluation_params=["actual_output", "expected_output"],
    )
    rule_quality_metric = GEval(
        name="Rule Discovery Quality",
        criteria="Does the model correctly discover the hidden transformation rule from the examples and apply it accurately?",
        evaluation_params=["actual_output", "expected_output"],
    )

    print("Running DeepEval GEval...")
    results = evaluate(test_cases, [correctness_metric, rule_quality_metric], verbose=False)

    mlflow.log_metric("deepeval_answer_correctness", results[0].score)
    mlflow.log_metric("deepeval_rule_quality", results[1].score)

    print(f"\n=== EVALUATION RESULTS ===")
    print(f"Exact Match (manual):       {avg_exact_match:.4f}")
    print(f"DeepEval Correctness:       {results[0].score:.4f}")
    print(f"DeepEval Rule Quality:      {results[1].score:.4f}")
    print(f"Evaluation time:            {time.time() - start_time:.1f} seconds")

    mlflow.end_run()

    return {
        "exact_match": avg_exact_match,
        "deepeval_correctness": results[0].score,
        "deepeval_rule_quality": results[1].score,
    }

        
    

In [33]:
def evaluate_model(
    model_name: str,
    adapter_path: Optional[str] = None,
    val_size: int = 400,
    experiment_name: str = "wonderland_nemotron_reasoning"
):
    """Convenience wrapper"""
    model, tokenizer = load_model(model_name, adapter_path)
    suffix = "base" if adapter_path is None else adapter_path.split("/")[-1]
    return run_evaluation(model, tokenizer, val_size, experiment_name, run_name_suffix=suffix)

## Evaluate base model

In [21]:
# evaluate_model(
#     model_name= BASE_MODEL_PATH,
#     adapter_path=None,          # or "./adapters/sft_final"
#     val_size=len(valDF)
# )

In [22]:
 model, tokenizer = load_model(BASE_MODEL_PATH, None)

Loading model: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 (base)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

In [23]:
testPrompt = valDF.iloc[0]["prompt"]
testAns = valDF.iloc[0]["answer"]


In [24]:
# messages = [
#     {"role": "system", "content": "You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles. Analyze the examples carefully and output ONLY the final answer."},
#     {"role": "user", "content": testPrompt}
# ]



In [25]:
full_prompt = f"""<|system|>You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles. Analyze the examples carefully and output ONLY the final answer.</s>
<|user|>{testPrompt}</s>
<|assistant|>"""

# Tokenize as string → tensor
tokenized = tokenizer(
    full_prompt,
    return_tensors="pt",
    add_special_tokens=True
)

input_ids = tokenized.input_ids.to(model.device)
attention_mask = tokenized.attention_mask.to(model.device)

# === Generate ===
outputs = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=256,
    do_sample=False,
    num_beams=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


In [29]:
# # Decode and extract answer
# === CLEAN DECODING (new) ===
full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

# Extract only assistant response
if "<|assistant|>" in full_output:
    pred = full_output.split("<|assistant|>")[-1]
else:
    pred = full_output

# Remove common special tokens
special_tokens = ["<|im_end|>", "<|endoftext|>", "</s>", "<|eot_id|>", "<|im_start|>"]
for token in special_tokens:
    pred = pred.replace(token, "").strip()

pred = pred.strip()   # final clean

In [30]:
full_output

"<|system|>You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles. Analyze the examples carefully and output ONLY the final answer.</s>\n<|user|>In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01001000 -> 10100100\n00011100 -> 11001110\n01001001 -> 10110100\n00011111 -> 11111111\n01000100 -> 01100010\n01010110 -> 01001011\n11000101 -> 00110010\n10010101 -> 00011010\n00000100 -> 01000010\n\nNow, determine the output for: 00101111</s>\n<|assistant|>10110100<|im_end|>"

In [31]:
pred

'10110100'

###
### Full SFT (Supervised Fine-Tuning) script for NVIDIA Nemotron challenge.
#### Uses real train.csv + modular functions from evaluation.py
###

In [ ]:
# ==================== FORMAT FOR GRPO ====================
def format_grpo_example(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    
    chat_prompt = f"""<|system|>You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.
    Analyze the given examples carefully, figure out the secret rule, and apply it precisely to the new input.</s>
<|user|>{prompt}</s>
<|assistant|>"""
    
    return {
        "prompt": chat_prompt,
        "ground_truth": answer
    }

In [ ]:
# Create dataset
records = [format_grpo_example(row) for _, row in trainDF.iterrows()]
dataset = Dataset.from_list(records)

In [ ]:
type(records),  len(records) , records[0]

In [ ]:
dataset

In [ ]:
# def load_model(
#     modelName: str ,
#     adapter_path: Optional[str] = None):
#     pass
